# 🧠 NLP-Based Early Detection of Depression and Anxiety Indicators
### Fine-tuned DistilBERT + SHAP/LIME Explainability
---
**Project Specification 54** | Domain: NLP / ML / Mental Health Analytics  
**UN SDG Goal 3** – Good Health and Well-being (Target 3.4)

---
### 📋 Dataset Summary:
| # | File | Columns | Label |
|---|------|---------|-------|
| 1 | reddit_posts_comments_Anxiety.csv | Post Title, Post Body, Comment | → Anxiety |
| 2 | stressed_anxious_cleaned.csv | Text, is_stressed/anxious | 1 = Anxiety |
| 3 | Suicide_Detection.csv | text, class | suicide / non-suicide |
| 4 | reddit_comments_clean.csv | clean_text, label | 0=Neutral / 1=Depression |

### 🏷️ Final Label Schema (4-class):
- **0 = Neutral** — no distress indicators
- **1 = Anxiety** — anxiety / stress indicators
- **2 = Depression** — depressive language patterns
- **3 = Suicide Risk** — suicide ideation language

> ⚠️ **Ethical Notice:** This system is a research prototype for early-warning screening **only**. It does **NOT** provide medical diagnosis.

---
## ⚙️ PHASE 1: Environment Setup

In [ ]:
# CELL 1-A: Install all required libraries
!pip install transformers==4.40.0 datasets torch scikit-learn shap lime \
             nltk spacy textblob matplotlib seaborn plotly \
             wordcloud imbalanced-learn accelerate evaluate -q

!python -m spacy download en_core_web_sm -q

import nltk
for pkg in ['stopwords','wordnet','punkt','averaged_perceptron_tagger','omw-1.4']:
    nltk.download(pkg, quiet=True)

print('All libraries installed successfully!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 22.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 51.7 MB/s eta 0:00:00


In [ ]:
# CELL 1-B: Import all libraries
import os, re, warnings, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
from wordcloud import WordCloud

# ML
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_recall_fscore_support
)

# Deep Learning
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    AdamW,
    get_linear_schedule_with_warmup
)

# Explainability
import shap
from lime.lime_text import LimeTextExplainer
from IPython.display import display, HTML

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Imports complete | Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## 📂 PHASE 2: Dataset Loading
> Upload all 4 dataset files to your Google Drive first, then set the paths below.

In [ ]:
# CELL 2-A: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

In [ ]:
# CELL 2-B: Set dataset file paths
# -------- UPDATE THESE PATHS TO MATCH YOUR DRIVE FOLDER --------
BASE_PATH = '/content/drive/MyDrive/mental_project/'   # <-- change this to your folder

PATH_ANXIETY_REDDIT   = BASE_PATH + 'reddit_posts_comments_Anxiety.csv'
PATH_STRESSED_ANXIOUS = BASE_PATH + 'stressed_anxious_cleaned.csv'
PATH_SUICIDE          = BASE_PATH + 'Suicide_Detection.csv'
PATH_REDDIT_COMMENTS  = BASE_PATH + 'reddit_comments_clean.csv'
# ----------------------------------------------------------------

# Check files exist
for name, path in [
    ('Anxiety Reddit',   PATH_ANXIETY_REDDIT),
    ('Stressed/Anxious', PATH_STRESSED_ANXIOUS),
    ('Suicide Detection',PATH_SUICIDE),
    ('Reddit Comments',  PATH_REDDIT_COMMENTS),
]:
    status = 'FOUND' if os.path.exists(path) else 'NOT FOUND - check path!'
    print(f'  {name:22s}: {status}')

In [ ]:
# CELL 2-C: Load all four datasets

# Dataset 1: Reddit Anxiety Posts (14,824 rows, no label -> assign Anxiety)
df1 = pd.read_csv(PATH_ANXIETY_REDDIT)
print(f'Dataset 1 shape: {df1.shape} | Cols: {df1.columns.tolist()}')

# Dataset 2: Stressed/Anxious (3,999 rows, label=1)
df2 = pd.read_csv(PATH_STRESSED_ANXIOUS)
print(f'Dataset 2 shape: {df2.shape} | Cols: {df2.columns.tolist()}')
print(f'  Labels: {df2["is_stressed/anxious"].value_counts().to_dict()}')

# Dataset 3: Suicide Detection (text, class: suicide/non-suicide)
df3 = pd.read_csv(PATH_SUICIDE)
# Handle possible extra unnamed index column
if df3.columns[0].startswith('Unnamed'):
    df3 = df3.iloc[:, 1:]
print(f'Dataset 3 shape: {df3.shape} | Cols: {df3.columns.tolist()}')
print(f'  Labels: {df3["class"].value_counts().to_dict()}')

# Dataset 4: Reddit Comments Clean (clean_text, label: 0/1)
df4 = pd.read_csv(PATH_REDDIT_COMMENTS)
print(f'Dataset 4 shape: {df4.shape} | Cols: {df4.columns.tolist()}')
print(f'  Labels: {df4["label"].value_counts().to_dict()}')

---
## 🧹 PHASE 3: Text Preprocessing

In [ ]:
# CELL 3-A: Build preprocessing pipeline
lemmatizer  = WordNetLemmatizer()
STOP_WORDS  = set(stopwords.words('english'))
# Keep negation words - critical for mental health text!
KEEP_WORDS  = {'no','not',"n't",'never','nobody','nothing','neither',
               'nor','nowhere','cannot','barely','hardly','scarcely'}
STOP_WORDS  = STOP_WORDS - KEEP_WORDS

def clean_text(text: str) -> str:
    """Full NLP preprocessing pipeline for social media mental health text."""
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)           # remove URLs
    text = re.sub(r'\[removed\]|\[deleted\]', '', text)    # Reddit artifacts
    text = re.sub(r'<.*?>', '', text)                       # HTML tags
    text = text.encode('ascii', 'ignore').decode('ascii')   # remove non-ASCII
    text = re.sub(r"[^a-z\s']", ' ', text)                 # keep letters+apostrophe
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [
        lemmatizer.lemmatize(t)
        for t in text.split()
        if t not in STOP_WORDS and len(t) > 1
    ]
    return ' '.join(tokens)

def get_sentiment(text: str) -> float:
    """TextBlob polarity: -1 (negative) to +1 (positive)."""
    try:
        return TextBlob(str(text)).sentiment.polarity
    except:
        return 0.0

# Test pipeline
sample = "I've been feeling so hopeless. Can't stop crying and nothing matters anymore."
print('Original :', sample)
print('Cleaned  :', clean_text(sample))
print('Sentiment:', get_sentiment(sample))

---
## 🔗 PHASE 4: Dataset Merging & Label Engineering

In [ ]:
# CELL 4-A: Unify all 4 datasets into one master DataFrame
# Label Schema: 0=Neutral | 1=Anxiety | 2=Depression | 3=Suicide Risk

parts = []

# --- Dataset 1: Reddit Anxiety Posts (no label → assign Anxiety=1) -----------
d1 = df1.copy()
d1['text'] = (
    d1['Post Title'].fillna('') + ' ' +
    d1['Post Body'].fillna('') + ' ' +
    d1['Comment'].fillna('')
).str.strip()
# Remove bot/mod messages
d1 = d1[~d1['Comment'].isin(['[removed]', '[deleted]'])]
d1['label_name'] = 'Anxiety'
d1['source']     = 'reddit_anxiety'
parts.append(d1[['text','label_name','source']])
print(f'Dataset 1: {len(d1):,} rows → Anxiety')

# --- Dataset 2: Stressed/Anxious (is_stressed=1 → Anxiety=1) -----------------
d2 = df2.copy().rename(columns={'Text':'text'})
d2['label_name'] = 'Anxiety'
d2['source']     = 'stressed_anxious'
parts.append(d2[['text','label_name','source']])
print(f'Dataset 2: {len(d2):,} rows → Anxiety')

# --- Dataset 3: Suicide Detection (suicide=3, non-suicide=0) -----------------
d3 = df3.copy()
# Find text column
text_col = [c for c in d3.columns if 'text' in c.lower()][0]
d3 = d3.rename(columns={text_col: 'text'})
d3['label_name'] = d3['class'].map({'suicide':'Suicide Risk','non-suicide':'Neutral'})
d3['source']     = 'suicide_detection'
parts.append(d3[['text','label_name','source']])
print(f'Dataset 3: {len(d3):,} rows → Suicide Risk + Neutral')

# --- Dataset 4: Reddit Comments (0=Neutral, 1=Depression) --------------------
d4 = df4.copy()
text_col4 = [c for c in d4.columns if 'text' in c.lower() or 'clean' in c.lower()][0]
d4 = d4.rename(columns={text_col4:'text'})
d4['label_name'] = d4['label'].map({0:'Neutral', 1:'Depression'})
d4['source']     = 'reddit_comments'
parts.append(d4[['text','label_name','source']])
print(f'Dataset 4: {len(d4):,} rows → Depression + Neutral')

# --- Merge & clean -----------------------------------------------------------
df = pd.concat(parts, ignore_index=True)
df = df.dropna(subset=['text','label_name'])
df = df[df['text'].str.strip().str.len() > 10]
df = df.reset_index(drop=True)

print(f'\nMaster dataset: {df.shape}')
print('\nClass distribution:')
print(df['label_name'].value_counts())

In [ ]:
# CELL 4-B: Apply text preprocessing (2-4 min)
print('Cleaning text — this takes 2-4 minutes on Colab...')
df['clean_text'] = df['text'].apply(clean_text)
df['sentiment']  = df['clean_text'].apply(get_sentiment)
df['text_len']   = df['clean_text'].apply(lambda x: len(x.split()))

# Remove empty after cleaning
df = df[df['clean_text'].str.strip().str.len() > 5].reset_index(drop=True)

print(f'Preprocessing complete!')
print(f'Total samples : {len(df):,}')
print(f'Avg text len  : {df["text_len"].mean():.1f} words')
df[['label_name','clean_text','sentiment','text_len']].head(5)

In [ ]:
# CELL 4-C: Exploratory Data Analysis (EDA)
COLORS = {'Neutral':'#2ecc71','Anxiety':'#e67e22','Depression':'#3498db','Suicide Risk':'#e74c3c'}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Dataset EDA — Mental Health NLP Project', fontsize=16, fontweight='bold')

# 1. Label distribution
lc = df['label_name'].value_counts()
axes[0,0].bar(lc.index, lc.values, color=[COLORS.get(l,'#95a5a6') for l in lc.index])
axes[0,0].set_title('Class Distribution')
for i,v in enumerate(lc.values): axes[0,0].text(i, v+200, str(v), ha='center', fontweight='bold')

# 2. Sentiment distribution by class
for lname, color in COLORS.items():
    sub = df[df['label_name']==lname]['sentiment']
    if len(sub) > 0:
        axes[0,1].hist(sub, bins=40, alpha=0.6, label=lname, color=color)
axes[0,1].set_title('Sentiment Distribution by Class')
axes[0,1].set_xlabel('Polarity')
axes[0,1].legend(fontsize=8)

# 3. Text length distribution
for lname, color in COLORS.items():
    sub = df[df['label_name']==lname]['text_len']
    if len(sub) > 0:
        axes[0,2].hist(sub.clip(0,300), bins=40, alpha=0.6, label=lname, color=color)
axes[0,2].set_title('Text Length Distribution')
axes[0,2].set_xlabel('Word Count (capped 300)')
axes[0,2].legend(fontsize=8)

# 4. Average sentiment by class
avg_s = df.groupby('label_name')['sentiment'].mean().sort_values()
axes[1,0].barh(avg_s.index, avg_s.values, color=[COLORS.get(l,'gray') for l in avg_s.index])
axes[1,0].axvline(0, color='black', ls='--', alpha=0.5)
axes[1,0].set_title('Average Sentiment by Class')

# 5. Source pie
sc = df['source'].value_counts()
axes[1,1].pie(sc.values, labels=sc.index, autopct='%1.1f%%',
              colors=['#3498db','#e74c3c','#2ecc71','#e67e22'])
axes[1,1].set_title('Data Source Distribution')

# 6. Box plot text length by class
present = [l for l in ['Neutral','Anxiety','Depression','Suicide Risk'] if l in df['label_name'].unique()]
axes[1,2].boxplot([df[df['label_name']==l]['text_len'].clip(0,400).values for l in present],
                  labels=present, patch_artist=True,
                  boxprops=dict(facecolor='lightblue'))
axes[1,2].set_title('Text Length by Class')
axes[1,2].set_ylabel('Word Count')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
 # CELL 4-D: Word clouds per class
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Word Clouds by Mental Health Category', fontsize=16, fontweight='bold')

for ax, (lname, cmap) in zip(axes.flat, [('Neutral','Greens'),('Anxiety','Oranges'),
                                           ('Depression','Blues'),('Suicide Risk','Reds')]):
    sub = df[df['label_name']==lname]['clean_text'].dropna()
    if len(sub) == 0:
        ax.set_title(f'{lname} (no data)'); ax.axis('off'); continue
    wc = WordCloud(width=600, height=400, background_color='white',
                   colormap=cmap, max_words=100
                  ).generate(' '.join(sub.values[:3000]))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'{lname}  (n={len(sub):,})', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CELL 4-E: Label encoding & train/val/test split

# Cap each class at 10k to manage Colab memory
MAX_PER_CLASS = 10_000
df_bal = (
    df.groupby('label_name', group_keys=False)
      .apply(lambda x: x.sample(min(len(x), MAX_PER_CLASS), random_state=SEED))
      .reset_index(drop=True)
)
print('Balanced class counts:')
print(df_bal['label_name'].value_counts())

# Build label maps (only for classes that exist)
present_labels  = sorted(df_bal['label_name'].unique(),
                         key=lambda x: {'Neutral':0,'Anxiety':1,'Depression':2,'Suicide Risk':3}[x])
LABEL2ID        = {l: i for i, l in enumerate(present_labels)}
ID2LABEL        = {i: l for l, i in LABEL2ID.items()}
NUM_LABELS      = len(LABEL2ID)
df_bal['label_id'] = df_bal['label_name'].map(LABEL2ID)

print(f'\nLabel mapping : {LABEL2ID}')
print(f'Num classes   : {NUM_LABELS}')

X = df_bal['clean_text'].values
y = df_bal['label_id'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

print(f'\nSplit sizes:')
print(f'  Train : {len(X_train):,}')
print(f'  Val   : {len(X_val):,}')
print(f'  Test  : {len(X_test):,}')

---
## 🤖 PHASE 5: DistilBERT Fine-Tuning

In [ ]:
# CELL 5-A: Tokenizer & PyTorch Dataset
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN    = 128    # good balance for Reddit posts
BATCH_SIZE = 32     # reduce to 16 if you get OOM errors

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class MentalHealthDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx]) if self.texts[idx] else ''
        enc  = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = MentalHealthDataset(X_train, y_train, tokenizer, MAX_LEN)
val_ds   = MentalHealthDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_ds  = MentalHealthDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Tokenizer: {MODEL_NAME}')
print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

In [ ]:
# CELL 5-B: Load model & configure optimizer/scheduler
EPOCHS       = 4
LR           = 2e-5
WARMUP_RATIO = 0.1

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(device)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = AdamW(model.parameters(), lr=LR, eps=1e-8)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

print(f'Model on: {device}')
print(f'Total params     : {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable params : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'Epochs: {EPOCHS} | LR: {LR} | Total steps: {total_steps}')

In [ ]:
# CELL 5-C: Training & evaluation functions
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()

def train_epoch(model, loader, optimizer, scheduler, scaler, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        labels= batch['label'].to(device)
        optimizer.zero_grad()
        with autocast():
            out  = model(input_ids=ids, attention_mask=mask, labels=labels)
            loss = out.loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
        correct    += (out.logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(device)
            mask  = batch['attention_mask'].to(device)
            labels= batch['label'].to(device)
            out   = model(input_ids=ids, attention_mask=mask, labels=labels)
            total_loss += out.loss.item()
            preds       = out.logits.argmax(1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss/len(loader), correct/total, all_preds, all_labels

print('Training functions defined!')

In [ ]:
# CELL 5-D: Run training loop
history = {'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[],'val_f1':[]}
best_val_f1    = 0.0
BEST_MODEL_PATH = 'best_model.pt'

print('Starting DistilBERT fine-tuning...\n')
print(f'{"Epoch":>5} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>8} | {"Val F1":>7}')
print('-' * 62)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc                    = train_epoch(model, train_loader, optimizer, scheduler, scaler, device)
    vl_loss, vl_acc, vl_preds, vl_true = eval_epoch(model, val_loader, device)
    vl_f1 = f1_score(vl_true, vl_preds, average='weighted')

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    history['val_f1'].append(vl_f1)

    marker = ' <-- best' if vl_f1 > best_val_f1 else ''
    print(f'{epoch:>5} | {tr_loss:>10.4f} | {tr_acc:>9.4f} | {vl_loss:>8.4f} | {vl_acc:>8.4f} | {vl_f1:>7.4f}{marker}')

    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        torch.save(model.state_dict(), BEST_MODEL_PATH)

print(f'\nTraining complete! Best Val F1: {best_val_f1:.4f}')

In [ ]:
# CELL 5-E: Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training Curves — DistilBERT Fine-Tuning', fontsize=14, fontweight='bold')
ep = range(1, EPOCHS + 1)

axes[0].plot(ep, history['train_loss'], 'b-o', label='Train')
axes[0].plot(ep, history['val_loss'],   'r-o', label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, history['train_acc'], 'b-o', label='Train')
axes[1].plot(ep, history['val_acc'],   'r-o', label='Val')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ep, history['val_f1'], 'g-o', label='Val F1')
axes[2].axhline(y=0.85, color='orange', ls='--', label='Target F1=0.85')
axes[2].set_title('Validation F1'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🔍 PHASE 6: Explainability — SHAP + LIME

In [ ]:
# CELL 6-A: Load best model & define prediction functions
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()
print('Best model loaded.')

class_names = [ID2LABEL[i] for i in range(NUM_LABELS)]

def predict_proba(texts):
    """Return softmax probabilities for list of texts."""
    if isinstance(texts, str): texts = [texts]
    enc = tokenizer(list(texts), max_length=MAX_LEN, padding=True,
                    truncation=True, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()

def predict_label(text: str) -> str:
    return ID2LABEL[np.argmax(predict_proba([text])[0])]

# Quick test
demo = "I can't sleep. Everything feels empty and I don't see a way forward."
probs = predict_proba([demo])[0]
print(f'\nDemo text: "{demo}"')
for i, p in enumerate(probs):
    bar = '█' * int(p*30) + '░' * (30-int(p*30))
    print(f'  {ID2LABEL[i]:15s} |{bar}| {p*100:5.1f}%')

In [ ]:
# CELL 6-B: LIME Explanations
lime_explainer = LimeTextExplainer(class_names=class_names, random_state=SEED)

def explain_lime(text: str, num_features: int = 12):
    """Explain a prediction with LIME and display highlighted text."""
    pred  = predict_label(text)
    pred_idx = LABEL2ID[pred]
    print(f'Input     : "{text[:100]}"')
    print(f'Predicted : {pred}')
    exp = lime_explainer.explain_instance(
        text, predict_proba,
        num_features=num_features,
        labels=[pred_idx],
        num_samples=300
    )
    display(HTML(exp.as_html(labels=[pred_idx])))
    return exp

# Run LIME on one example per class
lime_examples = [
    "I feel like I'm always on edge. My heart races and I panic for no reason.",
    "Life feels completely meaningless. I sleep all day and can't stop crying.",
    "I don't want to be here anymore. I've been thinking about ending it.",
    "Had a great day today! Finished my project and enjoyed a walk in the park."
]

for txt in lime_examples:
    print('\n' + '='*60)
    explain_lime(txt)

In [ ]:
# CELL 6-C: SHAP Explanations
print('Initialising SHAP explainer (~1 min)...')

shap_texts    = X_test[50:60].tolist()   # 10 samples
masker        = shap.maskers.Text(tokenizer=r'\W+')
shap_explainer = shap.Explainer(predict_proba, masker=masker, output_names=class_names)
shap_values    = shap_explainer(shap_texts, fixed_context=1)
print('SHAP values computed!')

In [ ]:
# CELL 6-D: Visualise SHAP
# Text-level explanations for first 3 samples
for i in range(min(3, len(shap_texts))):
    pred     = predict_label(shap_texts[i])
    pred_idx = LABEL2ID[pred]
    print(f'\nSample {i+1} — Predicted: {pred}')
    shap.plots.text(shap_values[i, :, pred_idx])

# Global feature importance
print('\nSHAP Global Feature Importance (mean |SHAP|):')
shap.plots.bar(shap_values[:, :, :].mean(0), max_display=15)

---
## 📊 PHASE 7: Evaluation + Dashboard

In [ ]:
# CELL 7-A: Full test set evaluation
print('Running inference on full test set...')
_, test_acc, test_preds, test_true = eval_epoch(model, test_loader, device)

test_f1_w = f1_score(test_true, test_preds, average='weighted')
test_f1_m = f1_score(test_true, test_preds, average='macro')
target_names = [ID2LABEL[i] for i in range(NUM_LABELS)]

print('\n' + '='*55)
print('  TEST SET RESULTS')
print('='*55)
print(f'  Accuracy     : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print(f'  F1 Weighted  : {test_f1_w:.4f}')
print(f'  F1 Macro     : {test_f1_m:.4f}')
print(f'  Target F1>0.85: {"ACHIEVED" if test_f1_w >= 0.85 else "Not reached"}')
print('='*55)
print('\nClassification Report:')
print(classification_report(test_true, test_preds, target_names=target_names))

In [ ]:
# CELL 7-B: Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Evaluation — DistilBERT Mental Health Classifier',
             fontsize=14, fontweight='bold')

cm = confusion_matrix(test_true, test_preds)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=target_names, yticklabels=target_names, ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalised)')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CELL 7-C: Per-class metrics bar chart
prec, rec, f1c, sup = precision_recall_fscore_support(
    test_true, test_preds, labels=list(range(NUM_LABELS)))

metrics_df = pd.DataFrame({
    'Class'    : target_names,
    'Precision': prec,
    'Recall'   : rec,
    'F1-Score' : f1c,
    'Support'  : sup
})

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(target_names))
w = 0.25
b1 = ax.bar(x-w, metrics_df['Precision'], w, label='Precision', color='#3498db')
b2 = ax.bar(x,   metrics_df['Recall'],    w, label='Recall',    color='#2ecc71')
b3 = ax.bar(x+w, metrics_df['F1-Score'],  w, label='F1-Score',  color='#e74c3c')
ax.axhline(0.85, color='orange', ls='--', alpha=0.7, label='Target F1=0.85')
ax.set_xticks(x); ax.set_xticklabels(target_names)
ax.set_ylim(0, 1.1); ax.legend(); ax.set_ylabel('Score')
ax.set_title('Per-Class Performance Metrics', fontsize=14, fontweight='bold')

for bars in [b1,b2,b3]:
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
                f'{b.get_height():.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print(metrics_df.to_string(index=False))

In [ ]:
# CELL 7-D: Interactive Risk Score Dashboard
RISK_WEIGHTS = {'Neutral':0.0,'Anxiety':0.33,'Depression':0.67,'Suicide Risk':1.0}
RISK_COLORS  = {'Neutral':'#2ecc71','Anxiety':'#f39c12','Depression':'#3498db','Suicide Risk':'#e74c3c'}

# Use 20 test samples
dash_texts  = X_test[:20].tolist()
dash_probs  = predict_proba(dash_texts)
dash_preds  = [ID2LABEL[np.argmax(p)] for p in dash_probs]
dash_conf   = [float(np.max(p)) for p in dash_probs]
dash_risk   = [sum(dash_probs[i][j] * RISK_WEIGHTS.get(ID2LABEL[j],0)
                    for j in range(NUM_LABELS)) for i in range(20)]

bar_colors = [RISK_COLORS.get(p,'#95a5a6') for p in dash_preds]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Risk Score (20 Samples)','Prediction Confidence',
                    'Class Probability Heatmap','Risk vs Confidence']
)

fig.add_trace(go.Bar(x=list(range(1,21)), y=dash_risk,
                     marker_color=bar_colors, name='Risk'), row=1, col=1)
fig.add_trace(go.Bar(x=list(range(1,21)), y=dash_conf,
                     marker_color='#9b59b6', name='Confidence'), row=1, col=2)
fig.add_trace(go.Heatmap(z=dash_probs, x=class_names,
                          y=[f'S{i+1}' for i in range(20)],
                          colorscale='RdYlGn_r'), row=2, col=1)
fig.add_trace(go.Scatter(x=dash_risk, y=dash_conf, mode='markers+text',
                          marker=dict(color=bar_colors, size=12),
                          text=[f'S{i+1}' for i in range(20)],
                          textposition='top center'), row=2, col=2)

fig.update_layout(title_text='Mental Health Early Warning Dashboard',
                  height=700, showlegend=False)
fig.show()
fig.write_html('dashboard.html')
print('Dashboard saved to dashboard.html')

In [ ]:
# CELL 7-E: Live Prediction Interface — try your own text!
def analyse(text: str) -> dict:
    """Full prediction pipeline for any input text."""
    cleaned   = clean_text(text)
    probs     = predict_proba([cleaned])[0]
    pred_idx  = int(np.argmax(probs))
    pred_name = ID2LABEL[pred_idx]
    sentiment = get_sentiment(text)
    risk      = sum(probs[i] * RISK_WEIGHTS.get(ID2LABEL[i],0) for i in range(NUM_LABELS))

    W = 30
    print('\n' + '='*62)
    print('  MENTAL HEALTH SCREENING RESULT  |  NOT A MEDICAL DIAGNOSIS')
    print('='*62)
    print(f'  Input      : "{text[:80]}"' + ('...' if len(text)>80 else ''))
    print(f'  Category   : {pred_name.upper()}')
    print(f'  Confidence : {probs[pred_idx]*100:.1f}%')
    print(f'  Risk Score : {risk:.2f}/1.00')
    print(f'  Sentiment  : {sentiment:+.3f}')
    print('\n  Class Probabilities:')
    for i in range(NUM_LABELS):
        filled = int(probs[i]*W)
        bar = '█'*filled + '░'*(W-filled)
        print(f'  {ID2LABEL[i]:15s} |{bar}| {probs[i]*100:5.1f}%')
    print('='*62)
    return {'label':pred_name,'confidence':float(probs[pred_idx]),'risk':float(risk)}


# -------- TRY YOUR OWN TEXT HERE ----------------------------------------
my_texts = [
    "My heart is racing all the time and I keep catastrophising about everything.",
    "I've lost interest in everything I used to love. I feel empty inside.",
    "I don't want to be here anymore. I've been thinking about ending it all.",
    "Just got promoted at work! Feeling really happy and motivated today."
]

for t in my_texts:
    analyse(t)

In [ ]:
# CELL 7-F: Save model & results to Drive
SAVE_DIR = BASE_PATH + 'mental_health_model'
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(f'{SAVE_DIR}/label_mappings.json', 'w') as f:
    json.dump({'label2id': LABEL2ID, 'id2label': ID2LABEL,
               'risk_weights': RISK_WEIGHTS}, f, indent=2)

results = {
    'model'          : MODEL_NAME,
    'epochs'         : EPOCHS,
    'max_len'        : MAX_LEN,
    'test_accuracy'  : round(test_acc, 4),
    'f1_weighted'    : round(test_f1_w, 4),
    'f1_macro'       : round(test_f1_m, 4),
    'best_val_f1'    : round(best_val_f1, 4),
    'target_achieved': bool(test_f1_w >= 0.85),
    'num_classes'    : NUM_LABELS,
    'train_size'     : len(X_train),
    'test_size'      : len(X_test),
}
with open(f'{SAVE_DIR}/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Model, tokenizer and results saved!')
print(json.dumps(results, indent=2))

In [ ]:
# CELL 7-G: Final Summary
print(f"""
  Model        : {MODEL_NAME}
  Classes      : {list(ID2LABEL.values())}
  Train samples: {len(X_train):,}
  Test samples : {len(X_test):,}

  Test Accuracy    : {test_acc:.4f}  ({test_acc*100:.1f}%)
  F1 Weighted      : {test_f1_w:.4f}
  F1 Macro         : {test_f1_m:.4f}
  Target F1 > 0.85 : {'ACHIEVED' if test_f1_w >= 0.85 else 'Not reached - try EPOCHS=6'}

  Outputs generated:
    eda_overview.png       — Dataset EDA charts
    wordclouds.png         — Per-class word clouds
    training_curves.png    — Loss / Accuracy / F1 curves
    confusion_matrix.png   — Confusion matrices
    per_class_metrics.png  — Precision/Recall/F1 per class
    dashboard.html         — Interactive risk dashboard
    best_model.pt          — Best checkpoint
    mental_health_model/   — HuggingFace model + tokenizer
    results.json           — Evaluation summary

  ETHICAL DISCLAIMER:
  This is a research screening prototype ONLY.
  It does NOT replace clinical care. If you or someone
  you know is struggling, please contact a professional.
""")
print('='*62)